In [1]:
import pandas as pd
import scipy
import statsmodels

print(pd.__version__, scipy.__version__, statsmodels.__version__)

2.0.3 1.11.1 0.14.0


In [2]:
df = pd.read_csv("../data/cookie_cats.csv")
print(df.shape)
df.head()

(90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [3]:
print("unique userid:", df["userid"].nunique())
print(df.isna().sum())
print(df["version"].value_counts())

unique userid: 90189
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64


In [4]:
print(df["sum_gamerounds"].describe())
print(df["sum_gamerounds"].nlargest(5))
print("zero rounds:", (df["sum_gamerounds"] == 0).sum())

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64
57702    49854
7912      2961
29417     2640
43671     2438
48188     2294
Name: sum_gamerounds, dtype: int64
zero rounds: 3994


In [5]:
pd.crosstab(df["retention_1"], df["retention_7"])

retention_7,False,True
retention_1,,
False,46437,3599
True,26971,13182


In [6]:
df.loc[57702]

userid            6390605
version           gate_30
sum_gamerounds      49854
retention_1         False
retention_7          True
Name: 57702, dtype: object

In [7]:
from scipy.stats import chisquare

observed = df["version"].value_counts().sort_index()
expected = [len(df) / 2, len(df) / 2]

stat, p_value = chisquare(f_obs=observed, f_exp=expected)
print(observed)
print("chi-square:", round(stat, 3), " p-value:", round(p_value, 5))

version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
chi-square: 6.902  p-value: 0.00861


In [8]:
df.groupby("version")[["retention_1", "retention_7"]].mean()

,retention_1,retention_7
version,,
gate_30,0.448188,0.190201
gate_40,0.442283,0.182000


In [9]:
from statsmodels.stats.proportion import proportions_ztest

n = df["version"].value_counts().sort_index()

for col in ["retention_1", "retention_7"]:
    retained = df.groupby("version")[col].sum().sort_index()
    rate = retained / n
    z, p = proportions_ztest(count=retained, nobs=n)
    diff_pp = (rate["gate_40"] - rate["gate_30"]) * 100
    print(col, "| gate_40 - gate_30:", round(diff_pp, 2), "pp | z:", round(z, 3), "| p:", round(p, 5))

retention_1 | gate_40 - gate_30: -0.59 pp | z: 1.784 | p: 0.07441
retention_7 | gate_40 - gate_30: -0.82 pp | z: 3.164 | p: 0.00155


In [10]:
import numpy as np

rng = np.random.default_rng(42)
g30 = df.loc[df["version"] == "gate_30", "retention_7"].to_numpy()
g40 = df.loc[df["version"] == "gate_40", "retention_7"].to_numpy()

boot_diffs = []
for _ in range(2000):
    s30 = rng.choice(g30, size=g30.size, replace=True).mean()
    s40 = rng.choice(g40, size=g40.size, replace=True).mean()
    boot_diffs.append((s40 - s30) * 100)

boot_diffs = np.array(boot_diffs)
low, high = np.percentile(boot_diffs, [2.5, 97.5])
print("mean diff:", round(boot_diffs.mean(), 2), "pp | 95% CI:", round(low, 2), "to", round(high, 2), "pp")

mean diff: -0.83 pp | 95% CI: -1.35 to -0.34 pp


In [11]:
from scipy.stats import norm

alpha, power = 0.05, 0.80
z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(power)
n30, n40 = n["gate_30"], n["gate_40"]

for col in ["retention_7", "retention_1"]:
    p0 = df.loc[df["version"] == "gate_30", col].mean()
    se = np.sqrt(p0 * (1 - p0) * (1 / n30 + 1 / n40))
    mde_pp = (z_alpha + z_power) * se * 100
    print(col, "| baseline:", round(p0 * 100, 2), "% | MDE:", round(mde_pp, 2),
          "pp | relative MDE:", round(mde_pp / (p0 * 100) * 100, 1), "%")

retention_7 | baseline: 19.02 % | MDE: 0.73 pp | relative MDE: 3.8 %
retention_1 | baseline: 44.82 % | MDE: 0.93 pp | relative MDE: 2.1 %


In [12]:
mask = df["userid"] != 6390605
rounds = df[mask].groupby("version")["sum_gamerounds"]
print("median:\n", rounds.median())
print("mean:\n", rounds.mean().round(2))

median:
 version
gate_30    17.0
gate_40    16.0
Name: sum_gamerounds, dtype: float64
mean:
 version
gate_30    51.34
gate_40    51.30
Name: sum_gamerounds, dtype: float64


In [13]:
bins = [-1, 0, 29, 99, df["sum_gamerounds"].max()]
labels = ["0", "1-29", "30-99", "100+"]
df["rounds_band"] = pd.cut(df["sum_gamerounds"], bins=bins, labels=labels)

seg = df.groupby(["rounds_band", "version"], observed=True)["retention_7"].agg(["count", "mean"])
seg["mean"] = (seg["mean"] * 100).round(2)
seg

count   mean
rounds_band version              
0           gate_30   1937   0.83
            gate_40   2057   0.63
1-29        gate_30  26107   4.51
            gate_40  26819   4.18
30-99       gate_30  10488  27.84
            gate_40  10265  25.67
100+        gate_30   6168  71.14
            gate_40   6348  71.03